# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 3: HUẤN LUYỆN, ĐÁNH GIÁ & TRIỂN KHAI MÔ HÌNH (MODEL TRAINING, EVALUATION & DEPLOYMENT)

> **Mục tiêu chính:**
> - Huấn luyện và đánh giá 5-Fold Stratified CV trên 3 thuật toán: **Naive Bayes**, **Logistic Regression ** và **Linear SVM**.
> - Tìm tham số tối ưu bằng `GridSearchCV`.
> - Đánh giá Hold-out Test Set
> - Giải thích 15 từ khóa báo thư spam mạnh nhất và đóng gói mô hình sẵn sàng sử dụng thực tế.

## Khai báo Thư viện & Nạp Dữ liệu Đã Xử lý từ Notebook 2
Nạp các thư viện , dữ liệu `train_test_data.joblib` và cấu trúc `preprocessor.joblib`.

In [ ]:
# Khai báo thư viện & Nạp dữ liệu
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

data_path = os.path.join('..', 'data', 'processed', 'train_test_data.joblib')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'processed', 'train_test_data.joblib')

prep_path = os.path.join('..', 'models', 'preprocessor.joblib')
if not os.path.exists(prep_path):
    prep_path = os.path.join('models', 'preprocessor.joblib')

data = joblib.load(data_path)
X_train, X_test, y_train, y_test = data['X_train'], data['X_test'], data['y_train'], data['y_test']
base_preprocessor = joblib.load(prep_path)
print(f"Nạp thành công: Train={len(X_train):,} mẫu, Test={len(X_test):,} mẫu")

---
## Mục 6: Huấn luyện mô hình & Tối ưu siêu tham số

### 6.1 Thiết lập 3 mô hình & Kiểm định Chéo Phân tầng (5-Fold Stratified CV)
Đánh giá chéo 5-Fold cho 3 thuật toán trên tập Train:
1. **Naive Bayes:** Thuật toán xác suất đơn giản, tốc độ huấn luyện cực nhanh.
2. **Logistic Regression:** Đo lường trọng số từng đặc trưng, tự động triệt tiêu trọng số từ khóa vô dụng.
3. **Support Vector Machine(Linear SVM):** Thuật toán tìm đường ranh giới phân tách rõ ràng nhất giữa thư thường và thư rác.

In [ ]:
# 3 Pipeline & 5-Fold Stratified CV trên tập Train
from sklearn.base import clone

models = {
    'naive_bayes': MultinomialNB(),
    'logistic_regression': LogisticRegression(penalty='l1', solver='liblinear', random_state=RANDOM_SEED, max_iter=1000),
    'linear_svm': CalibratedClassifierCV(estimator=LinearSVC(random_state=RANDOM_SEED, max_iter=3000, dual='auto'), method='sigmoid', cv=3)
}

pipelines = {name: Pipeline([('preprocessor', clone(base_preprocessor)), ('classifier', model)]) for name, model in models.items()}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_summary = []

for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs=-1)
    cv_summary.append({
        'Mô hình': name,
        'Accuracy (%)': f"{scores['test_accuracy'].mean()*100:.2f} ± {scores['test_accuracy'].std()*100:.2f}",
        'Precision (%)': f"{scores['test_precision'].mean()*100:.2f} ± {scores['test_precision'].std()*100:.2f}",
        'Recall (%)': f"{scores['test_recall'].mean()*100:.2f} ± {scores['test_recall'].std()*100:.2f}",
        'F1-Score (%)': f"{scores['test_f1'].mean()*100:.2f} ± {scores['test_f1'].std()*100:.2f}",
        'ROC-AUC': f"{scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}"
    })

display(pd.DataFrame(cv_summary))

**Đúc kết kết quả thi thử (5-Fold CV):**
- Cả 3 mô hình đều đạt điểm F1-Score trên 92% với độ lệch chuẩn rất nhỏ (±0.06% - 0.24%), chứng tỏ khả năng học bài rất ổn định.
- **Linear SVM** (98,91%) và **Logistic Regression** (98,34%) thể hiện độ chính xác vượt trội.
- **Naive Bayes** (92,76%) chạy rất nhanh nhưng độ bao quát (Recall đạt 87,61%) còn thấp, vẫn bỏ sót nhiều thư rác.

### 6.2 Tinh chỉnh tìm siêu tham số tối ưu (GridSearchCV)
Thử nghiệm tự động các tổ hợp tham số (vốn từ vựng của TF-IDF, số từ chọn lọc của Chi-Square và độ nhạy của thuật toán) để đưa từng mô hình lên mức điểm cao nhất.

In [ ]:
# Tinh chỉnh siêu tham số GridSearchCV
models_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else os.path.join('models')
os.makedirs(models_dir, exist_ok=True)

param_grids = {
    'naive_bayes': {
        'preprocessor__text__tfidf__max_features': [3000, 4000, 5000],
        'preprocessor__text__select__k': [500, 1000, 1500, 2000],
        'classifier__alpha': [0.1, 0.5, 1.0]
    },
    'logistic_regression': {
        'preprocessor__text__tfidf__max_features': [3000, 4000, 5000],
        'preprocessor__text__select__k': [500, 1000, 1500, 2000],
        'classifier__C': [0.5, 1.0, 5.0]
    },
    'linear_svm': {
        'preprocessor__text__tfidf__max_features': [3000, 4000, 5000],
        'preprocessor__text__select__k': [500, 1000, 1500, 2000],
        'classifier__estimator__C': [0.5, 1.0, 3.0]
    }
}

tuned_models = {}
for name, pipe in pipelines.items():
    gs = GridSearchCV(pipe, param_grid=param_grids[name], cv=cv, scoring='f1', n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)
    tuned_models[name] = gs.best_estimator_
    print(f"{name}: Best F1={gs.best_score_*100:.2f}% | Params: {gs.best_params_}")
    joblib.dump(gs.best_estimator_, os.path.join(models_dir, f"{name}.joblib"))

print(f"Đã lưu 3 mô hình tối ưu vào {models_dir}/")

**Đúc kết sau khi tinh chỉnh siêu tham số:**
- Cả 3 mô hình đều đạt phong độ cao nhất khi mở rộng tối đa vốn từ: lấy **5.000 từ phổ biến** (`max_features = 5000`) và giữ lại **2.000 từ tinh hoa nhất** (`select__k = 2000`).
- Điểm F1-Score thi thử trên tập Train đều tăng rõ rệt[cite: 5]:
  - **Linear SVM:** Đạt đỉnh **99,20%**.
  - **Logistic Regression:** Đạt **99,06%**.
  - **Naive Bayes:** Đạt **95,83%**.
- Toàn bộ 3 mô hình tối ưu đã được lưu thành các tệp nhị phân `.joblib` vào thư mục `models/`.

---
## Mục 7: Đánh giá Mô hình trên Tập Kiểm tra (Model Evaluation) & Phân tích lỗi

### 7.1 Đánh giá Độc lập trên Tập Kiểm tra (Hold-out Test Set Evaluation)
**Mục tiêu:** Đo lường 5 chỉ số hiệu năng trên 6.532 mẫu kiểm tra độc lập chưa từng xuất hiện trong huấn luyện.

In [ ]:
# Đánh giá trên Hold-out Test Set (6.532 mẫu)
test_results = []
for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    test_results.append({
        'Mô hình': name,
        'Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
        'FP': int(fp),
        'FN': int(fn)
    })

display(pd.DataFrame(test_results))

**Đúc kết từ Bảng Đánh giá Tập Kiểm tra Độc lập (Hold-out Test Set - 6.532 mẫu):**
- **Support Vector Machine (Linear SVM):** Quán quân toàn diện với **F1-Score 99.44%** (Accuracy 99.46%, Precision 99.45%, Recall 99.42%, ROC-AUC 0.9996). So với cấu hình baseline ban đầu (F1 99.31%), bước tinh chỉnh siêu tham số và kết hợp 6 đặc trưng miền đã đẩy hiệu năng lên mức tối ưu.
- **Logistic Regression (Chuẩn hóa L1):** Đạt sát nút với **F1-Score 99.29%** (Accuracy 99.33%, Precision 99.26%, Recall 99.33%, ROC-AUC 0.9994; baseline ban đầu F1 99.16%), cơ chế phạt L1 triệt tiêu hiệu quả các đặc trưng nhiễu, mang lại khả năng diễn giải toán học mẫu mực.
- **Naive Bayes:** Đạt **F1-Score 95.78%** (Accuracy 96.11%, ROC-AUC 0.9978; baseline ban đầu F1 92.70%). Precision đạt mức rất cao (99.11%), tuy nhiên Recall thấp hơn (92.68%) do giả định các từ khóa độc lập có điều kiện bị vi phạm trong thực tế khi các từ ngữ lừa đảo thường xuất hiện đồng thời.

**Bảng xếp hạng trên đề thi thật (6.532 mẫu Test):**
- **Top 1 - Linear SVM:** F1 đạt **99,45%** (Accuracy 99,48%, Precision 99,49%, Recall 99,42%, ROC-AUC 0.9997).
- **Top 2 - Logistic Regression:** F1 đạt **99,33%** (Accuracy 99,36%, ROC-AUC 0.9995).
- **Top 3 - Naive Bayes:** Đạt F1 **95,98%** (Accuracy 96,30%, ROC-AUC 0.9982). Dù độ chuẩn xác khi báo spam rất cao (Precision 99,31%), nhưng độ bao phủ chỉ đạt 92,87% do vẫn để lọt lưới nhiều thư rác.

### 7.2 Phân tích các ca đoán sai (Confusion Matrix)
Kiểm tra thực tế 2 loại lỗi quan trọng:
- **Bắt nhầm thư quan trọng (FP - False Positive):** Email công việc/thư thường bị đánh nhãn rác.
- **Bỏ sót thư rác (FN - False Negative):** Thư rác lừa đảo lọt qua bộ lọc vào hộp thư chính.

In [ ]:
# Trực quan hóa Ma trận nhầm lẫn cho cả 3 mô hình
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles = {'naive_bayes': 'Naive Bayes', 'logistic_regression': 'Logistic Regression (L1)', 'linear_svm': 'Support Vector Machine (SVM)'}

for idx, (name, model) in enumerate(tuned_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Dự đoán Ham', 'Dự đoán Spam'], yticklabels=['Thực tế Ham', 'Thực tế Spam'])
    axes[idx].set_title(titles[name], fontweight='bold')
plt.tight_layout()
plt.show()

**Đúc kết phân tích sai số thực tế (trên 6.532 mẫu Test):**
- **Linear SVM an toàn nhất:** Chỉ bắt nhầm **16 thư thường** thành rác (dưới 0,5% tổng số thư thường) và chỉ để lọt **18 thư rác**, hạn chế tối đa rủi ro làm thất lạc email công việc quan trọng của người dùng.
- **Logistic Regression cân bằng:** Bắt nhầm **20 thư thường** và bỏ sót **22 thư rác**, duy trì độ ổn định rất cao.
- **Naive Bayes bỏ sót nhiều:** Dù chỉ bắt nhầm 20 thư thường, mô hình lại bỏ lọt tới **222 thư rác** vào hộp thư (do giả định các từ khóa xuất hiện độc lập không hoàn toàn đúng với thực tế các chiêu trò spam).

### 7.3 Giải thích mô hình: Top 15 từ khóa báo hiệu thư rác mạnh nhất
Mở mô hình tốt nhất (Linear SVM) để bóc tách các từ ngữ có trọng số nghi vấn spam lớn nhất.

In [ ]:
# Trích xuất Top 15 từ khóa Spam mạnh nhất từ Linear SVM
best_model = tuned_models['linear_svm']
prep = best_model.named_steps['preprocessor']
clf = best_model.named_steps['classifier']

# Lấy toàn bộ 1.506 tên đặc trưng từ ColumnTransformer và loại bỏ tiền tố text__ / num__
feature_names = [f.replace('text__', '').replace('num__', '') for f in prep.get_feature_names_out()]
feature_names = np.array(feature_names)

# Trích xuất trọng số trung bình từ các bộ ước lượng CalibratedClassifierCV
weights = np.mean([c.estimator.coef_[0] for c in clf.calibrated_classifiers_], axis=0)

top15_idx = np.argsort(weights)[-15:][::-1]
top15_df = pd.DataFrame({'Từ khóa': feature_names[top15_idx], 'Trọng số': weights[top15_idx]})

plt.figure(figsize=(10, 5))
sns.barplot(data=top15_df, x='Trọng số', y='Từ khóa', palette='Reds_r')
plt.title("Top 15 Từ khóa Báo hiệu Thư Rác Mạnh nhất (Linear SVM)", fontweight='bold')
plt.tight_layout()
plt.show()
display(top15_df)

**Đúc kết về các từ ngữ đặc trưng của thư rác:**
- 15 từ khóa có trọng số lớn nhất phản ánh chính xác 3 chủ đề thư rác phổ biến trong thực tế:
  1. *Cảnh báo & Dịch vụ giả mạo:* `custom alert`, `cnn com`, `alerts custom`, `suite`.
  2. *Biệt dược & Sức khỏe nhạy cảm:* `healthcare`, `sex`, `enhancement`, `dimension`.
  3. *Lừa đảo quà tặng & Giảm giá sốc:* `ecard`, `lowest`, `watches`.
- Việc các cụm 2 từ (như `custom alert`, `cnn com`) lọt vào nhóm dẫn đầu chứng minh lựa chọn tách cụm từ (`ngram_range=(1, 2)`) ở Phần 2 là hoàn toàn chính xác.

### 7.4 Đóng gói mô hình và hàm suy luận trực tiếp
- Lưu mô hình tốt nhất thành `best_model.joblib`.
- Xuất tệp thông số kết quả `metadata.json`.
- Xây dựng hàm `predict_email_spam()` tích hợp sẵn các bước làm sạch và trích xuất 7 đặc trưng đầu vào để sẵn sàng kiểm tra email mới tức thì.

In [ ]:
# Lưu best_model, metadata.json và hàm suy luận
from datetime import datetime
import re, html
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

models_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else os.path.join('models')
os.makedirs(models_dir, exist_ok=True)

# 1. Lưu best_model.joblib
best_path = os.path.join(models_dir, 'best_model.joblib')
joblib.dump(best_model, best_path)

# 2. Xuất metadata.json động từ test_results
best_metrics = next(item for item in test_results if item['Mô hình'] == 'linear_svm')
metadata = {
    "project_name": "Practice 1: Classifying Spam Emails",
    "team": "SpamGuard-ML Team",
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_model": "Support Vector Machine (Linear SVM)",
    "test_metrics": {
        "accuracy_percent": best_metrics['Accuracy (%)'],
        "precision_percent": best_metrics['Precision (%)'],
        "recall_percent": best_metrics['Recall (%)'],
        "f1_score_percent": best_metrics['F1-Score (%)']
    },
    "models_comparison": [
        {
            "Model": "Support Vector Machine (Linear SVM)" if r['Mô hình'] == 'linear_svm' else ("Logistic Regression (Chuẩn hóa L1)" if r['Mô hình'] == 'logistic_regression' else "Naive Bayes"),
            "Accuracy (%)": r['Accuracy (%)'],
            "Precision (%)": r['Precision (%)'],
            "Recall (%)": r['Recall (%)'],
            "F1-Score (%)": r['F1-Score (%)'],
            "ROC-AUC": r['ROC-AUC'],
            "FP": r.get('FP', 0),
            "FN": r.get('FN', 0)
        }
        for r in test_results
    ],
    "pipeline_steps": [
        "preprocessor (ColumnTransformer: TF-IDF + Chi-Square + MinMaxScaler)",
        "classifier (Calibrated LinearSVC)"
    ],
    "top_15_keywords": [
        {"rank": i + 1, "keyword": feat, "weight": round(float(w), 4)}
        for i, (feat, w) in enumerate(zip(top15_df['Từ khóa'], top15_df['Trọng số']))
    ]
}
with open(os.path.join(models_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"Đã lưu best_model.joblib và metadata.json tại {models_dir}/")

# 3. Hàm suy luận độc lập chuẩn 7 thuộc tính đầu vào
STOP_WORDS = set(ENGLISH_STOP_WORDS)
def clean_inference_text(text):
    if not isinstance(text, str) or not text.strip(): return ''
    text = re.sub(r'[^a-zA-Z\s]', ' ', re.sub(r'http[s]?://\S+|www\.\S+', ' ', re.sub(r'<[^>]+>', ' ', html.unescape(text))))
    return ' '.join([w for w in text.lower().split() if w not in STOP_WORDS and len(w) > 1])

def predict_email_spam(email_text, subject=None, model_path=best_path):
    if subject is None:
        subj_match = re.search(r'(?i)^\s*subject:\s*(.*)$', email_text, re.MULTILINE)
        if subj_match:
            subject = subj_match.group(1).strip()
            body = re.sub(r'(?i)^\s*subject:\s*.*$', '', email_text, count=1, flags=re.MULTILINE).strip()
        else:
            lines = [l.strip() for l in email_text.splitlines() if l.strip()]
            subject = lines[0] if lines else ''
            body = ' '.join(lines[1:]) if len(lines) > 1 else ''
    else:
        body = email_text

    raw_combined = f"{subject} {body}".strip()
    total_len = len(raw_combined)
    caps_count = sum(1 for c in raw_combined if c.isupper())
    caps_ratio = (caps_count / total_len) if total_len > 0 else 0.0
    exclamation_count = raw_combined.count('!')
    dollar_count = raw_combined.count('$')
    clean = clean_inference_text(raw_combined)
    text_len = len(clean)
    has_re_prefix = 1 if bool(re.search(r'^\s*re:', str(subject), re.IGNORECASE)) else 0
    subject_exclamation = 1 if '!' in str(subject) else 0

    if not clean and exclamation_count == 0 and dollar_count == 0 and caps_ratio == 0:
        return {'label': 0, 'label_name': 'Ham', 'spam_probability_percent': 0.0, 'note': 'Văn bản không có tín hiệu nguy cơ'}

    df_sample = pd.DataFrame([{
        'text_clean': clean,
        'caps_ratio': float(caps_ratio),
        'exclamation_count': int(exclamation_count),
        'dollar_count': int(dollar_count),
        'text_len': int(text_len),
        'has_re_prefix': int(has_re_prefix),
        'subject_exclamation': int(subject_exclamation)
    }])

    loaded = joblib.load(model_path)
    pred = loaded.predict(df_sample)[0]
    prob = loaded.predict_proba(df_sample)[0][1] if hasattr(loaded, 'predict_proba') else (0.95 if pred == 1 else 0.05)
    return {
        'label': int(pred),
        'label_name': 'Spam' if pred == 1 else 'Ham',
        'spam_probability_percent': round(prob * 100, 2)
    }

print("Test Ham:", predict_email_spam("Dear Team, please review the project schedule before tomorrow meeting.", subject="Meeting Schedule"))
print("Test Spam:", predict_email_spam("CONGRATULATIONS! You won a $1,000 Walmart Gift Card! Click here to claim.", subject="PRIZE ALERT!"))

**Tổng kết toàn bộ dự án:**
1. **Quy trình chuẩn hóa khép kín:** Hoàn thiện trọn vẹn luồng học máy từ Làm sạch dữ liệu (Phần 1), Kỹ thuật đặc trưng chống rò rỉ (Phần 2), đến Huấn luyện, Tối ưu và Đánh giá chi tiết (Phần 3).
2. **Mô hình xuất sắc:** Thuật toán **Linear SVM** đạt hiệu năng cao nhất với **F1-Score 99,45%**, phân loại cực kỳ chính xác và hiếm khi bắt nhầm email quan trọng.
3. **Sẵn sàng triển khai:** Mô hình tối ưu cùng hàm suy luận và tệp cấu hình `metadata.json` đã được lưu trữ hoàn chỉnh, sẵn sàng kết nối trực tiếp vào giao diện Web Streamlit (`app.py`) hoặc công cụ dòng lệnh.